**Quantum superposition and measurement.**
A qubit in superposition does not have a definite value until it is measured.
Before measurement its state is a normalized complex linear combination
$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ with $|\alpha|^2 + |\beta|^2 = 1$.
Measurement collapses the state: the Born rule gives $P(0) = |\alpha|^2$ and
$P(1) = |\beta|^2$.

A single `AerSimulator` backend is shared across all cells.
Each circuit uses `save_statevector` at intermediate steps to watch
the state move on the Bloch sphere before measurement collapses it.

In [ ]:
"""superposition.ipynb"""

# Cell 01 - Imports and shared AerSimulator backend

import numpy as np
from IPython.display import display
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import (
    plot_bloch_multivector,
    plot_distribution,
    plot_histogram,
)
from qiskit_aer import AerSimulator

from qis101_utils import as_latex

backend = AerSimulator()

print(f"Backend name    : {backend.name}")
print(f"Aer version     : {backend.configuration().backend_version}")
print(f"Max qubits      : {backend.configuration().n_qubits}")
print(f"Max shots       : {backend.configuration().max_shots:,}")
print(f"Simulation type : {backend.configuration().simulator}")
methods = backend.available_methods()
if methods:
    print("Available methods:")
    for method in methods:
        print(f"  {method}")


---
**Cell 02 - One qubit initialized to $|1\rangle$, then measured in the Z basis.**
The qubit starts at the south pole of the Bloch sphere.
An X gate flips it to the north pole ($|0\rangle$).
Since both states are Z-basis eigenstates, measurement is deterministic:
the result is always `0` with probability 1.

`plot_distribution` displays the y-axis as "quasi-probability", which despite
its name has nothing to do with the quantum mechanical quasi-probability
distributions (such as the Wigner function) that can go negative.
Qiskit simply means the raw shot counts divided by the total number of shots -
ordinary empirical frequencies normalized to sum to 1.
Here with a deterministic outcome the bar for `0` will reach exactly 1.0.

In [ ]:
# Cell 02 - One qubit on Z basis: initialize |1>, apply X, measure
# X|1> = |0>, so measurement always returns 0

qc = QuantumCircuit(1)
qc.initialize([0, 1], 0)
qc.save_statevector("sv1")
qc.x(0)
qc.save_statevector("sv2")
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))

display(plot_distribution(result.get_counts(qc)))

---
**Cell 03 - One qubit not on the Z basis.**
The initial state $[0.5,\;\sqrt{3}/2]^T$ is a superposition with
$P(0) = 0.5^2 = 25\%$ and $P(1) = (\sqrt{3}/2)^2 = 75\%$.
The X gate swaps the amplitudes, giving
$P(0) = 75\%$ and $P(1) = 25\%$ after the gate.
Measurement is now **probabilistic** - the outcome varies from shot to shot.

In [ ]:
# Cell 03 - One qubit NOT on Z basis: initialize superposition, apply X, measure
# [0.5, sqrt(3)/2]: P(0)=25%, P(1)=75%
# After X:          P(0)=75%, P(1)=25%

qc = QuantumCircuit(1)
qc.initialize([0.5, np.sqrt(3) / 2], 0)
qc.save_statevector("sv1")
qc.x(0)
qc.save_statevector("sv2")
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))

display(plot_distribution(result.get_counts(qc)))

---
**Cell 04 - One Pete Box: Hadamard applied to $|1\rangle$.**
The Hadamard gate maps the south pole ($|1\rangle$) to the $-X$ axis
of the Bloch sphere, creating the equal superposition $|-\rangle = (|0\rangle - |1\rangle)/\sqrt{2}$.
Measurement in the Z basis then gives 0 or 1 with equal probability (50/50).

The "Pete Box" framing comes from the pedagogical device of imagining a black box
that can place a coin into superposition - here H is that box.

In [ ]:
# Cell 04 - One Pete Box: H|1> = |-> = (|0>-|1>)/sqrt(2)
# Measurement: 50% '0', 50% '1'

qc = QuantumCircuit(1)
qc.initialize([0, 1], 0)
qc.save_statevector("sv1")
qc.h(0)
qc.save_statevector("sv2")
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))

display(plot_distribution(result.get_counts(qc)))

---
**Cell 05 - Two Pete Boxes: $H^2 = I$.**
Applying H twice returns the qubit to its original state.
This demonstrates that H is **its own inverse**: $H^2 = I$.
On the Bloch sphere the two Hadamard gates perform two $180°$ rotations
about the same axis, composing to a full $360°$ rotation (the identity).
Measurement after the second H is again deterministic.

In [ ]:
# Cell 05 - Two Pete Boxes: H @ H = I, so H|1> then H -> back to |1>
# Measurement: always '1'

qc = QuantumCircuit(1)
qc.initialize([0, 1], 0)
qc.save_statevector("sv1")
qc.h(0)
qc.save_statevector("sv2")
qc.h(0)
qc.save_statevector("sv3")
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_bloch_multivector(sv3))

display(plot_distribution(result.get_counts(qc)))

---
**Cell 06 - Gate equivalence: $HZH = X$.**
This circuit applies $H \to Z \to H$ to $|1\rangle$ without measuring,
demonstrating the identity $HZH = X$ verified algebraically in `quantum_gates.ipynb`.
The final statevector should equal $X|1\rangle = |0\rangle$,
confirming that the three-gate sequence has the same effect as a single X gate.

In [ ]:
# Cell 06 - Gate equivalence (no measurement): HZH|1> = X|1> = |0>
# Demonstrates the identity HZH = X

qc = QuantumCircuit(1)
qc.initialize([0, 1], 0)
qc.save_statevector("sv1")
qc.h(0)
qc.save_statevector("sv2")
qc.z(0)
qc.save_statevector("sv3")
qc.h(0)
qc.save_statevector("sv4")

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend)).result()
sv1 = result.data(0)["sv1"]
sv2 = result.data(0)["sv2"]
sv3 = result.data(0)["sv3"]
sv4 = result.data(0)["sv4"]

display(as_latex(sv1, prefix=r"\mathbf{Statevector\;1}="))
display(plot_bloch_multivector(sv1))
display(as_latex(sv2, prefix=r"\mathbf{Statevector\;2}="))
display(plot_bloch_multivector(sv2))
display(as_latex(sv3, prefix=r"\mathbf{Statevector\;3}="))
display(plot_bloch_multivector(sv3))
display(as_latex(sv4, prefix=r"\mathbf{Statevector\;4}="))
display(plot_bloch_multivector(sv4))

---
**Cell 07 - Insufficient number of shots.**
The initial state $[0.5,\;\sqrt{3}/2]^T$ has amplitudes $\alpha = 0.5$ and
$\beta = \sqrt{3}/2$, so the Born rule gives:

$$P(0) = |\alpha|^2 = 0.5^2 = 0.25 \qquad P(1) = |\beta|^2 = \left(\frac{\sqrt{3}}{2}\right)^2 = 0.75$$

The X gate then swaps the two amplitudes, flipping the probabilities:
$P(0) = 75\%$ and $P(1) = 25\%$.

With only 10 shots the histogram is noisy and may not clearly show this
$75\% / 25\%$ split.
This is quantum mechanics' analogue of the law of large numbers:
relative frequencies approach probabilities as the number of trials grows.

In [ ]:
# Cell 07 - Superposition measurement with 10 shots: noisy histogram
# State [0.5, sqrt(3)/2] after X: P(0)=75%, P(1)=25%
# With only 10 shots the result varies widely between runs

qc = QuantumCircuit(1)
qc.initialize([0.5, np.sqrt(3) / 2], 0)
qc.x(0)
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend), shots=10).result()
display(plot_histogram(result.get_counts(qc)))

---
**Cell 08 - The law of large numbers and the Born rule.**
With 100,000 shots the distribution converges to the theoretical probabilities
$P(0) = |\alpha|^2 = 0.25$ and $P(1) = |\beta|^2 = 0.75$ predicted by the Born rule.
This is quantum mechanics' analogue of the law of large numbers:
relative frequencies approach probabilities as the number of trials grows.

`plot_histogram` is used here instead of `plot_distribution`.
The difference is what appears on the y-axis:
`plot_distribution` divides the counts by the total number of shots and
displays normalized frequencies labeled "quasi-probability" (values between 0 and 1),
while `plot_histogram` displays the raw integer counts directly
(here you should see approximately 75,000 zeros and 25,000 ones).
The raw counts make the connection to the law of large numbers more concrete:
you can directly see how many times each outcome actually occurred
rather than reading off a normalized fraction.

In [ ]:
# Cell 08 - Same circuit with 100,000 shots: converges to Born rule probabilities
# Expected: ~75,000 counts of '0' and ~25,000 counts of '1'

qc = QuantumCircuit(1)
qc.initialize([0.5, np.sqrt(3) / 2], 0)
qc.x(0)
qc.measure_all()

display(qc.draw(output="mpl"))

result = backend.run(transpile(qc, backend), shots=100_000).result()
display(plot_histogram(result.get_counts(qc)))